<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>GPT on TinyStories</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/ebf621d/examples/gpt_tinystories.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/gpt_tinystories.ipynb)

</div>

---

Training a character-level GPT on [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) using [Ion](https://github.com/auxeno/ion). Use a GPU or TPU runtime for training.

In [ ]:
# !pip install ion-nn optax plotly tqdm

In [2]:
# CUDA 12 GPU
# !pip install --upgrade jax[cuda12]

# CUDA 13 GPU
# !pip install --upgrade jax[cuda13]

# TPU
# !pip install --upgrade jax[tpu]

In [3]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio
from tqdm import tqdm

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## Download & Prepare Data

We download the [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) train and validation splits, then build a simple character-level tokenizer over printable ASCII plus a special `<|endoftext|>` token (98 tokens total). The entire corpus is encoded into a flat `uint8` array for fast random-access batching.

In [4]:
from pathlib import Path
from urllib.request import Request, urlopen

CACHE_DIR = Path.home() / ".cache" / "ion" / "tinystories"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-"
)

for split in ("train", "valid"):
    dest = CACHE_DIR / f"tinystories-{split}.txt"
    if dest.exists():
        print(f"Skipping {split}, already cached.")
        continue
    url = f"{BASE_URL}{split}.txt"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request) as response, dest.open("wb") as f:
        total = int(response.headers.get("Content-Length", 0)) or None
        with tqdm(total=total, desc=f"Downloading {split}", unit="B", unit_scale=True) as pbar:
            while chunk := response.read(1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))

# Printable ASCII (same 97 chars as fable) + <|endoftext|> special token = 98 tokens
CHAR_TO_ID: dict[str, int] = {
    "\n": 0, "\t": 1, " ": 2, "!": 3, '"': 4, "#": 5, "$": 6, "%": 7,
    "&": 8, "'": 9, "(": 10, ")": 11, "*": 12, "+": 13, ",": 14, "-": 15,
    ".": 16, "/": 17, "0": 18, "1": 19, "2": 20, "3": 21, "4": 22, "5": 23,
    "6": 24, "7": 25, "8": 26, "9": 27, ":": 28, ";": 29, "<": 30, "=": 31,
    ">": 32, "?": 33, "@": 34, "A": 35, "B": 36, "C": 37, "D": 38, "E": 39,
    "F": 40, "G": 41, "H": 42, "I": 43, "J": 44, "K": 45, "L": 46, "M": 47,
    "N": 48, "O": 49, "P": 50, "Q": 51, "R": 52, "S": 53, "T": 54, "U": 55,
    "V": 56, "W": 57, "X": 58, "Y": 59, "Z": 60, "[": 61, "\\": 62, "]": 63,
    "^": 64, "_": 65, "`": 66, "a": 67, "b": 68, "c": 69, "d": 70, "e": 71,
    "f": 72, "g": 73, "h": 74, "i": 75, "j": 76, "k": 77, "l": 78, "m": 79,
    "n": 80, "o": 81, "p": 82, "q": 83, "r": 84, "s": 85, "t": 86, "u": 87,
    "v": 88, "w": 89, "x": 90, "y": 91, "z": 92, "{": 93, "|": 94, "}": 95,
    "~": 96,
}
EOT_TOKEN = "<|endoftext|>"
EOT_ID = 97
VOCAB_SIZE = 98

ID_TO_CHAR: dict[int, str] = {v: k for k, v in CHAR_TO_ID.items()}
ID_TO_CHAR[EOT_ID] = EOT_TOKEN


def tokenize(text: str) -> list[int]:
    """Tokenize text to token IDs, recognizing <|endoftext|> as a special token."""
    ids = []
    i = 0
    while i < len(text):
        if text[i:].startswith(EOT_TOKEN):
            ids.append(EOT_ID)
            i += len(EOT_TOKEN)
        else:
            ids.append(CHAR_TO_ID[text[i]])
            i += 1
    return ids


def detokenize(ids) -> str:
    """Detokenize token IDs back to text."""
    return "".join(ID_TO_CHAR[int(i)] for i in ids)

Skipping train, already cached.
Skipping valid, already cached.


## Clean stories
Drop any containing characters outside our vocabulary, then encode the cleaned text into a flat `uint8` token array.

In [5]:
valid_chars = set(CHAR_TO_ID.keys())


def load_and_tokenize(split: str) -> np.ndarray:
    """Read a TinyStories split, drop stories with out-of-vocab chars, return uint8 tokens."""
    path = CACHE_DIR / f"tinystories-{split}.txt"
    text = path.read_text(encoding="utf-8")

    # Split on <|endoftext|> delimiter, keeping the delimiter for encoding
    stories = text.split(EOT_TOKEN)

    kept, dropped = 0, 0
    token_ids: list[int] = []
    for story in tqdm(stories, desc=f"Tokenizing {split}", unit=" story"):
        if not story:
            continue
        if set(story).issubset(valid_chars):
            token_ids.extend(tokenize(story + EOT_TOKEN))
            kept += 1
        else:
            dropped += 1

    print(f"  kept {kept:,} stories, dropped {dropped:,}")
    return np.array(token_ids, dtype=np.uint8)


train_tokens = load_and_tokenize("train")
valid_tokens = load_and_tokenize("valid")

print(f"\nTrain tokens: {len(train_tokens):,}")
print(f"Valid tokens: {len(valid_tokens):,}")
print(f"Vocab size:   {VOCAB_SIZE}")

Tokenizing train: 100%|██████████| 2119719/2119719 [04:53<00:00, 7224.12 story/s]


  kept 1,989,604 stories, dropped 130,115


Tokenizing valid: 100%|██████████| 21990/21990 [00:02<00:00, 7380.26 story/s]


  kept 20,755 stories, dropped 1,235

Train tokens: 1,789,763,146
Valid tokens: 18,142,893
Vocab size:   98


## Define GPT Model

A minimal GPT built entirely from Ion's existing building blocks:

- `nn.Embedding` for the token embedding lookup
- `nn.sinusoidal()` for fixed positional encodings (not trainable)
- `nn.TransformerBlock` with `causal=True` for masked self-attention + FFN
- `nn.LayerNorm` as the final normalization

The output projection reuses the embedding weights (weight tying) via `x @ self.embed.w.T`.

**Mixed-precision strategy (bfloat16):**
- Master weights stored in **float32** for precise gradient accumulation
- Cast to **bfloat16** via `model.astype` at the start of the forward pass for fast matmuls
- Cast logits back to **float32** before softmax for numerical stability
- Optimizer states remain in **float32** (optax default)

In [6]:
from jaxtyping import Array, Float, Int, PRNGKeyArray

DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 6
SEQ_LEN = 256

w_init = jax.nn.initializers.truncated_normal(0.02)


class TransformerBlock(nn.Module):
    """Pre-norm causal self-attention + FFN block."""

    att: nn.SelfAttention
    norm_att: nn.LayerNorm
    norm_ff: nn.LayerNorm
    ff_1: nn.Linear
    ff_2: nn.Linear

    def __init__(self, dim: int, num_heads: int, *, key: PRNGKeyArray) -> None:
        key_att, key_ff1, key_ff2 = jax.random.split(key, 3)

        self.att = nn.SelfAttention(dim, num_heads, causal=True, w_init=w_init, key=key_att)
        self.norm_att = nn.LayerNorm(dim)
        self.norm_ff = nn.LayerNorm(dim)
        self.ff_1 = nn.Linear(dim, 4 * dim, bias=False, w_init=w_init, key=key_ff1)
        self.ff_2 = nn.Linear(4 * dim, dim, bias=False, w_init=w_init, key=key_ff2)

    def __call__(self, x: Float[Array, "b s d"]) -> Float[Array, "b s d"]:
        residual = x
        x = self.att(self.norm_att(x))
        x = x + residual

        residual = x
        x = self.ff_2(jax.nn.gelu(self.ff_1(self.norm_ff(x))))
        x = x + residual

        return x


class GPT(nn.Module):
    embed: nn.Embedding
    pos_enc: jax.Array  # fixed sinusoidal, not a Param
    blocks: tuple[TransformerBlock, ...]
    norm: nn.LayerNorm

    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        dim: int = DIM,
        num_heads: int = NUM_HEADS,
        num_layers: int = NUM_LAYERS,
        seq_len: int = SEQ_LEN,
        *,
        key: PRNGKeyArray,
    ) -> None:
        keys = jax.random.split(key, num_layers + 1)

        self.embed = nn.Embedding(vocab_size, dim, key=keys[0])
        self.pos_enc = nn.sinusoidal(seq_len, dim)

        self.blocks = tuple(
            TransformerBlock(dim, num_heads, key=keys[i + 1])
            for i in range(num_layers)
        )

        self.norm = nn.LayerNorm(dim)

    def __call__(self, x: Int[Array, "b s"]) -> Float[Array, "b s vocab"]:
        x = self.embed(x) + self.pos_enc[:x.shape[1]]
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = x @ self.embed.w.T  # weight tying
        return logits


model = GPT(key=jax.random.key(0))
print(f"Parameters: {model.num_params:,}")
print(f"Embed dtype: {model.embed.w.dtype}")
model

Parameters: 1,195,520
Embed dtype: float32


GPT(
  embed=Embedding(
    w=Param(f32[98, 128], trainable=True),
  ),
  pos_enc=f32[256, 128],
  blocks=(
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      ff_1=Linear(
        w=Param(f32[128, 512], trainable=True),
        b=None,
      ),
      ff_2=Linear(
        w=Param(f32[512, 128], trainable=True),
        b=None,
      ),
    ),
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      ff_1=Linear(
        w=Param(f32[128, 512], trainable=True),
        b=None,
      ),
      ff_2=Linear(
        w=Param(f32[512, 128], trainable=True),
        b=None,
      ),
    ),
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      ff_1=Linear(
        w=Param(f32[128, 512], trainable=True),
        b=None,
      ),
      ff_2=Linear(
        w=Param(f32[512, 128], trainable=True),
        b=None,
      ),
    ),
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      ff_1=Linear(
        w=Param(f32[128, 512], trainable=True),
        b=None,
      ),
      ff_2=Linear(
        w=Param(f32[512, 128], trainable=True),
        b=None,
      ),
    ),
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      ff_1=Linear(
        w=Param(f32[128, 512], trainable=True),
        b=None,
      ),
      ff_2=Linear(
        w=Param(f32[512, 128], trainable=True),
        b=None,
      ),
    ),
    TransformerBlock(
      att=SelfAttention(
        w_qkv=Param(f32[128, 3, 4, 32], trainable=True),
        w_out=Param(f32[4, 32, 128], trainable=True),
        b_out=None,
        causal=True,
      ),
      norm_att=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=True),
        eps=1e-05,
      ),
      norm_ff=LayerNorm(
        scale=Param(f32[128], trainable=True),
        b=Param(f32[128], trainable=Tr

## Training

We train with AdamW and a warmup-cosine learning rate schedule. Batches are formed by sampling random contiguous windows from the token stream, with inputs and targets offset by one position.

This model trains in around 10 mins on Google Colab on a v6e1 TPU at around ~3 million tokens/s.

In [7]:
from collections import deque

BATCH_SIZE = 64
LEARNING_RATE = 5e-4
COMPUTE_DTYPE = jnp.bfloat16
NUM_EPOCHS = 1
EVAL_EVERY = 100


def build_batch(num_tokens: int, batch_size: int, seq_len: int, *, key: jax.Array) -> Int[Array, "b s+1"]:
    """Sample random contiguous window indices; inputs and targets offset by 1."""
    starts = jax.random.randint(
        key, shape=(batch_size,), minval=0, maxval=num_tokens - seq_len - 1
    )
    return starts[:, None] + jnp.arange(seq_len + 1)


def loss_fn(
    model: GPT,
    inputs: Int[Array, "b s"],
    targets: Int[Array, "b s"],
) -> Float[Array, ""]:
    # Cast f32 master weights to bf16 for the forward pass
    model = model.astype(COMPUTE_DTYPE)
    logits = model(inputs)

    # Cast back to float32 for numerical stability before softmax
    logits = logits.astype(jnp.float32)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
    return loss


@jax.jit
def train_step(
    model: GPT,
    optimizer: ion.Optimizer,
    inputs: Int[Array, "b s"],
    targets: Int[Array, "b s"],
) -> tuple[GPT, ion.Optimizer, Float[Array, ""]]:
    loss, grads = jax.value_and_grad(loss_fn)(model, inputs, targets)
    model, optimizer = optimizer.update(model, grads)
    return model, optimizer, loss


@jax.jit
def eval_step(
    model: GPT,
    inputs: Int[Array, "b s"],
    targets: Int[Array, "b s"],
) -> Float[Array, ""]:
    return loss_fn(model, inputs, targets)


epoch_steps = len(train_tokens) // (BATCH_SIZE * SEQ_LEN)
total_steps = epoch_steps * NUM_EPOCHS

learning_rate = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LEARNING_RATE,
    warmup_steps=int(0.01 * total_steps),
    decay_steps=total_steps,
    end_value=0.1 * LEARNING_RATE,
)

model = GPT(key=jax.random.key(0))
optimizer = ion.Optimizer(optax.adamw(learning_rate), model)

# Move token arrays to device once
train_tokens = jnp.asarray(train_tokens)
valid_tokens = jnp.asarray(valid_tokens)

print(f"Steps per epoch: {epoch_steps:,}")
print(f"Total steps:     {total_steps:,}")

Steps per epoch: 109,238
Total steps:     109,238


In [8]:
rng = jax.random.key(0)
train_losses = []

for epoch in range(NUM_EPOCHS):
    rolling = deque(maxlen=100)

    desc = f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    pbar = tqdm(range(epoch_steps), desc=desc)

    for step in pbar:
        rng, key_batch = jax.random.split(rng)
        windows = train_tokens[build_batch(len(train_tokens), BATCH_SIZE, SEQ_LEN, key=key_batch)]
        inputs, targets = windows[:, :-1], windows[:, 1:]

        model, optimizer, loss = train_step(model, optimizer, inputs, targets)
        rolling.append(loss.item())
        train_losses.append(loss.item())

        if step % EVAL_EVERY == 0:
            rng, key_val = jax.random.split(rng)
            val_windows = valid_tokens[build_batch(len(valid_tokens), BATCH_SIZE, SEQ_LEN, key=key_val)]
            val_inputs, val_targets = val_windows[:, :-1], val_windows[:, 1:]
            val_loss = eval_step(model, val_inputs, val_targets)
            pbar.set_postfix(
                train_loss=f"{sum(rolling) / len(rolling):.3f}",
                val_loss=f"{val_loss.item():.3f}",
            )

Epoch 1/1: 100%|██████████| 109238/109238 [10:24<00:00, 175.00it/s, train_loss=0.639, val_loss=0.646]


## Loss Visualization

In [9]:
# Smooth with a rolling average then subsample
window = 50
smoothed = np.convolve(train_losses, np.ones(window) / window, mode="valid")
smoothed = smoothed[::max(1, len(smoothed) // 2000)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=smoothed,
    mode="lines",
    name="Training loss (smoothed)",
    line=dict(width=3, color="#636EFA"),
    opacity=0.9,
))
fig.update_layout(
    title="Training loss",
    xaxis_title="Step",
    yaxis_title="Cross-entropy loss",
    height=400,
    template="plotly_white",
)
fig.show()

## Text Generation

Autoregressive sampling with temperature scaling. We feed the prompt through the model, then repeatedly sample the next token from the predicted distribution and append it to the context.

In [10]:
@jax.jit
def sample_next(
    model: GPT,
    context: Int[Array, " s"],
    temperature: Float[Array, ""],
    *,
    key: jax.Array,
) -> Int[Array, ""]:
    """Run forward pass and sample the next token."""
    logits = model(context[None, -SEQ_LEN:])
    next_logits = logits[0, -1, :].astype(jnp.float32) / jnp.maximum(temperature, 1e-5)
    return jax.random.categorical(key, next_logits)


def generate(
    model: GPT,
    prompt: str,
    max_tokens: int = 500,
    temperature: float = 0.7,
    seed: int = 42,
) -> None:
    """Autoregressively sample and stream text from the model."""
    rng = jax.random.key(seed)
    context = jnp.array(tokenize(prompt), dtype=jnp.int32)
    temperature = jnp.float32(temperature)

    print(prompt, end="", flush=True)

    for _ in range(max_tokens):
        rng, key = jax.random.split(rng)
        next_token = sample_next(model, context, temperature, key=key)

        if next_token.item() == EOT_ID:
            break

        print(detokenize([next_token.item()]), end="", flush=True)
        context = jnp.concatenate([context, next_token[None]])

    print()

In [11]:
prompts = [
    "Once upon a time,",
    "The dog ran to the park and",
    "One sunny morning, Tom found a",
]

for prompt in prompts:
    generate(model, prompt)
    print("-" * 60)

Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine, but one day she found a big box high. She started to look for her name and she found it under a screen. She was so happy!
Lily said, "Look, my box! I found a big tree!" She put on her shoes and said, "Oh no! I found it!" Her mom said, "Why don't you both look at it? It will be fun to see it together."
Lily was sad that she couldn't go out and it was stuck in a tree. But she knew that if she had to take it everywhere 
------------------------------------------------------------
The dog ran to the park and got closer. The dog saw the slide and the sandbox and the books. They ran to the slide and climbed up.
Ben and Lily were scared. They saw a big truck with a loud square on the floor. The snow was rotting too. The snow fell over the fence. The snowman did not like the snowman.
"Look, the snowman is alive!" Lily said. "The snowman is looking for me!"
"Run, Run!" Tom said. He grabbed a stick a